# Week 1 — Baseline evaluation (B0 zero-shot, B1 3-shot)

Runtime → Change runtime type → **T4 GPU**.

Upload the `finetuning_slm` folder to Google Drive (`MyDrive/finetuning_slm`) first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/finetuning_slm
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt

Sanity check: generator unit tests (should print OK).

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -3

Smoke test on 5 items per task before the full run.

In [ ]:
!python eval/run_eval.py --run-name smoke --limit 5 --tasks csbench gen_os mbpp

**B0: zero-shot baseline** (~400 CS-Bench + 100 OS problems + 200 MBPP).

In [ ]:
!python eval/run_eval.py --run-name B0_zeroshot --tasks csbench gen_os mbpp

**B1: 3-shot baseline.**

In [ ]:
!python eval/run_eval.py --run-name B1_3shot --fewshot 3 --tasks csbench gen_os mbpp

Optional: Spider (download the Spider zip from https://yale-lily.github.io/spider, unzip to `spider/`).

In [ ]:
# !python eval/run_eval.py --run-name B0_zeroshot_spider --tasks spider --spider-dir spider

## Results

In [ ]:
import pandas as pd, json
df = pd.read_csv('eval/results/summary.csv')
display(df[df.run != 'smoke'])
for run in ['B0_zeroshot', 'B1_3shot']:
    s = json.load(open(f'eval/results/{run}/summary.json'))['results']
    print(run, {t: {g: round(v['acc'], 3) for g, v in r['by_group'].items()} for t, r in s.items()})

Look at failures — they tell you what the training data must fix:

In [ ]:
rows = [json.loads(l) for l in open('eval/results/B0_zeroshot/gen_os.jsonl')]
for r in [r for r in rows if not r['correct']][:3]:
    print(r['prompt'][:400], '
--- MODEL ---
', r['output'][-600:], '
' + '=' * 80)